In [1]:
import cv2
import numpy as np
import pandas as pd
import datetime
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from sklearn.svm import SVC
import joblib
import os

2025-08-21 14:44:46.153079: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755787486.511731      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755787486.610337      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# STEP 1: Train Face Recognition Model (SVM)
# ===============================
def train_face_model(student_dataset_dir="student_dataset"):
    data = []
    labels = []
    label_dict = {}

    label_id = 0
    for student in os.listdir(student_dataset_dir):
        student_path = os.path.join(student_dataset_dir, student)
        if not os.path.isdir(student_path):
            continue
        label_dict[label_id] = student
        for img_name in os.listdir(student_path):
            img_path = os.path.join(student_path, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None: continue
            img = cv2.resize(img, (100, 100)).flatten()
            data.append(img)
            labels.append(label_id)
        label_id += 1

    data = np.array(data)
    labels = np.array(labels)

    svm = SVC(kernel='linear', probability=True)
    svm.fit(data, labels)

    joblib.dump(svm, "face_recognition_svm.pkl")
    joblib.dump(label_dict, "label_dict.pkl")
    print("Face Recognition Model Trained & Saved.")


In [3]:
# STEP 2: Train Emotion Model
# ===============================
def train_emotion_model(train_dir, test_dir):
    datagen = ImageDataGenerator(rescale=1./255)

    train_gen = datagen.flow_from_directory(
        train_dir, target_size=(96, 96),
        color_mode='grayscale', class_mode='categorical'
    )
    val_gen = datagen.flow_from_directory(
        test_dir, target_size=(96, 96),
        color_mode='grayscale', class_mode='categorical'
    )

    input_layer = Input(shape=(96, 96, 1))
    base = MobileNetV2(weights=None, include_top=False, input_shape=(96,96,3))
    x = tf.keras.layers.Conv2D(3, (3,3), padding="same")(input_layer)
    x = base(x)
    x = GlobalAveragePooling2D()(x)
    out = Dense(train_gen.num_classes, activation="softmax")(x)

    model = Model(inputs=input_layer, outputs=out)
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    model.fit(train_gen, validation_data=val_gen, epochs=20)

    model.save("emotion_model.h5")
    print("Emotion Model Trained & Saved.")